# Chinese Understanding Benchmark: Qwen3.5-4B vs Gemma E4B

This notebook evaluates and fine-tunes two small language models on standard Chinese language understanding datasets from CLUE.

Default models:
- `Qwen/Qwen3.5-4B`
- `google/gemma-4-E4B`

Default CLUE tasks:
- `afqmc`: sentence-pair semantic matching
- `tnews`: news title classification
- `cmnli`: natural language inference

Designed for a 32GB Apple Silicon Mac. Start with baseline evaluation first. Fine-tuning is optional and uses LoRA, not full fine-tuning.

## 1. Install dependencies

In [1]:
%pip install -U torch transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece protobuf

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

Some gated Google models require `huggingface-cli login` before loading. Keep sample counts small at first on a laptop.

In [2]:
import os, re, time, platform, gc
from typing import Dict, List, Any, Optional
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments

try:
    from peft import LoraConfig, PeftModel
    from trl import SFTTrainer
    PEFT_AVAILABLE = True
except Exception as e:
    print("PEFT/TRL import failed:", e)
    PEFT_AVAILABLE = False

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

Python: 3.11.9
Torch: 2.12.0
MPS available: True
CUDA available: False


In [3]:
MODELS = {
    "qwen35_4b": "Qwen/Qwen3.5-4B",
    "gemma_e4b": "google/gemma-4-E4B",
}

TASKS = ["afqmc", "tnews", "cmnli"]
SPLIT = "validation"
MAX_EVAL_SAMPLES = 200      # raise after smoke test
MAX_TRAIN_SAMPLES = 1000    # raise after smoke test
RESULTS_DIR = "results"
ADAPTERS_DIR = "adapters"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(ADAPTERS_DIR, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
DEVICE

'mps'

## 3. CLUE task definitions and prompts

This version prompts with Chinese verbal labels, then maps predictions back to the dataset's numeric label IDs for fair scoring.

In [4]:
# Chinese verbal labels for prompting, with robust mapping back to dataset IDs.
# Why: raw numeric IDs make generation brittle, especially for TNEWS and CMNLI.

TASK_SPECS = {
    "afqmc": {
        "label_id_to_name": {0: "不同", 1: "相同"},
        "label_name_to_id": {"不同": "0", "相同": "1"},
    },
    "cmnli": {
        # CLUE/HF CMNLI is commonly: 0=entailment, 1=neutral, 2=contradiction.
        "label_id_to_name": {0: "蕴含", 1: "中立", 2: "矛盾"},
        "label_name_to_id": {"蕴含": "0", "中立": "1", "矛盾": "2"},
    },
}

# TNEWS has two common ID styles:
# - original CLUE IDs: 100,101,102,103,104,106,107,108,109,110,112,113,114,115,116
# - Hugging Face ClassLabel indices: 0..14 in the same order
TNEWS_RAW_IDS = [100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 112, 113, 114, 115, 116]
TNEWS_NAMES = ["故事", "文化", "娱乐", "体育", "财经", "房产", "汽车", "教育", "科技", "军事", "旅游", "国际", "股票", "农业", "电竞"]
TNEWS_RAW_ID_TO_NAME = dict(zip(TNEWS_RAW_IDS, TNEWS_NAMES))
TNEWS_INDEX_TO_NAME = dict(enumerate(TNEWS_NAMES))
TNEWS_NAME_TO_RAW_ID = {name: str(raw_id) for raw_id, name in zip(TNEWS_RAW_IDS, TNEWS_NAMES)}
TNEWS_NAME_TO_INDEX = {name: str(i) for i, name in enumerate(TNEWS_NAMES)}


def get_label_names(task: str) -> List[str]:
    if task == "tnews":
        return TNEWS_NAMES
    return list(TASK_SPECS[task]["label_name_to_id"].keys())


def infer_label_id_style(task: str, ds) -> str:
    """Return the target ID style used by the loaded dataset split."""
    if task != "tnews":
        return "default"
    sample_n = min(100, len(ds))
    ids = {int(ds[i]["label"]) for i in range(sample_n)}
    if ids and max(ids) <= 14:
        return "tnews_index"      # HF ClassLabel style: 0..14
    return "tnews_raw"            # Original CLUE style: 100,101,...


def label_id_to_name(task: str, label_id: Any) -> str:
    label_id = int(label_id)
    if task == "tnews":
        if label_id in TNEWS_INDEX_TO_NAME:
            return TNEWS_INDEX_TO_NAME[label_id]
        return TNEWS_RAW_ID_TO_NAME.get(label_id, str(label_id))
    return TASK_SPECS[task]["label_id_to_name"].get(label_id, str(label_id))


def label_name_to_id(task: str, label_name: str, label_id_style: str = "default") -> str:
    if label_name == "__invalid__":
        return "__invalid__"
    if task == "tnews":
        if label_id_style == "tnews_index":
            return TNEWS_NAME_TO_INDEX.get(label_name, "__invalid__")
        return TNEWS_NAME_TO_RAW_ID.get(label_name, "__invalid__")
    return TASK_SPECS[task]["label_name_to_id"].get(label_name, "__invalid__")


def build_prompt(task: str, ex: Dict[str, Any]) -> str:
    labels = "、".join(get_label_names(task))

    if task == "afqmc":
        return f"""你是中文语义理解分类器。只能输出一个标签，不要解释。

任务：判断两个句子的语义是否相同。
可选标签：{labels}

句子1：{ex['sentence1']}
句子2：{ex['sentence2']}

答案："""

    if task == "cmnli":
        return f"""你是中文自然语言推理分类器。只能输出一个标签，不要解释。

任务：判断“假设”与“前提”的关系。
可选标签：{labels}

前提：{ex['sentence1']}
假设：{ex['sentence2']}

答案："""

    if task == "tnews":
        return f"""你是中文新闻标题分类器。只能输出一个标签，不要解释。

任务：判断新闻标题所属类别。
可选标签：{labels}

标题：{ex['sentence']}

答案："""

    raise ValueError(f"Unsupported task: {task}")


def get_gold_label(task: str, ex: Dict[str, Any]) -> str:
    return str(ex["label"])


def extract_label_name(task: str, text: str) -> str:
    text = str(text).strip()
    label_names = get_label_names(task)

    # Exact first.
    for label in label_names:
        if text == label:
            return label

    # Then prefix, useful when the model emits punctuation/newline after the label.
    for label in label_names:
        if text.startswith(label):
            return label

    # Then substring fallback, useful for outputs like “答案是：相同”.
    for label in sorted(label_names, key=len, reverse=True):
        if label in text:
            return label

    return "__invalid__"


## 4. Load CLUE datasets

In [5]:
def load_task_dataset(task: str, split: str = SPLIT, max_samples: Optional[int] = None):
    ds = load_dataset("clue", task, split=split)
    if max_samples is not None:
        ds = ds.select(range(min(max_samples, len(ds))))
    return ds

for task in TASKS:
    ds = load_task_dataset(task, SPLIT, 3)
    print("\n", task, ds)
    print(ds[0])


 afqmc Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3
})
{'sentence1': '双十一花呗提额在哪', 'sentence2': '里可以提花呗额度', 'label': 0, 'idx': 0}

 tnews Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 3
})
{'sentence': '江疏影甜甜圈自拍，迷之角度竟这么好看，美吸引一切事物', 'label': 2, 'idx': 0}

 cmnli Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3
})
{'sentence1': '新的权利已经足够好了', 'sentence2': '每个人都很喜欢最新的福利', 'label': 0, 'idx': 0}


## 5. Model loading and baseline evaluation

On a 32GB Mac, load/evaluate one model at a time. If memory pressure is high, lower `MAX_EVAL_SAMPLES` or restart the kernel between models.

In [6]:
def cleanup_memory():
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()


def load_causal_lm(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = dict(trust_remote_code=True)
    if DEVICE == "cuda":
        kwargs.update(torch_dtype=torch.float16, device_map="auto")
    elif DEVICE == "mps":
        kwargs.update(torch_dtype=torch.float16)
    else:
        kwargs.update(torch_dtype=torch.float32)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if DEVICE in ["mps", "cpu"]:
        model.to(DEVICE)
    model.eval()
    return tokenizer, model


@torch.no_grad()
def generate_answer(tokenizer, model, prompt: str, max_new_tokens: int = 4) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_model_on_task(model_key: str, model_id: str, task: str, split: str = SPLIT, max_samples: int = MAX_EVAL_SAMPLES):
    ds = load_task_dataset(task, split, max_samples)
    label_id_style = infer_label_id_style(task, ds)
    valid_ids = {label_name_to_id(task, name, label_id_style) for name in get_label_names(task)}

    tokenizer, model = load_causal_lm(model_id)
    rows = []
    start = time.time()

    for ex in tqdm(ds, desc=f"{model_key} / {task}"):
        prompt = build_prompt(task, ex)
        raw = generate_answer(tokenizer, model, prompt, max_new_tokens=4)
        pred_name = extract_label_name(task, raw)
        pred_id = label_name_to_id(task, pred_name, label_id_style)
        gold_id = get_gold_label(task, ex)
        gold_name = label_id_to_name(task, gold_id)

        rows.append({
            "model_key": model_key,
            "model_id": model_id,
            "task": task,
            "split": split,
            "gold_id": gold_id,
            "gold_name": gold_name,
            "pred_id": pred_id,
            "pred_name": pred_name,
            "raw_output": raw,
            "prompt": prompt,
        })

    elapsed = time.time() - start
    del model
    cleanup_memory()

    df = pd.DataFrame(rows)
    valid_pred = df["pred_id"].where(df["pred_id"].isin(valid_ids), "__invalid__")
    summary = {
        "model_key": model_key,
        "model_id": model_id,
        "task": task,
        "split": split,
        "samples": len(df),
        "accuracy": accuracy_score(df["gold_id"], valid_pred),
        "macro_f1": f1_score(df["gold_id"], valid_pred, average="macro", zero_division=0),
        "invalid_rate": float((valid_pred == "__invalid__").mean()),
        "seconds": elapsed,
        "samples_per_second": len(df) / max(elapsed, 1e-9),
        "label_id_style": label_id_style,
    }
    return df, summary


In [7]:
# Baseline evaluation. This can take a while on CPU/MPS.
baseline_rows, baseline_summaries = [], []
for model_key, model_id in MODELS.items():
    for task in TASKS:
        try:
            df_task, summary = evaluate_model_on_task(model_key, model_id, task)
            baseline_rows.append(df_task)
            baseline_summaries.append(summary)
            pd.DataFrame(baseline_summaries).to_csv(f"{RESULTS_DIR}/baseline_summary.csv", index=False)
            pd.concat(baseline_rows, ignore_index=True).to_csv(f"{RESULTS_DIR}/baseline_predictions.csv", index=False)
            print(summary)
        except Exception as e:
            print(f"FAILED: {model_key} / {task}: {type(e).__name__}: {e}")

baseline_summary_df = pd.DataFrame(baseline_summaries)
baseline_summary_df

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

qwen35_4b / afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'qwen35_4b', 'model_id': 'Qwen/Qwen3.5-4B', 'task': 'afqmc', 'split': 'validation', 'samples': 200, 'accuracy': 0.0, 'macro_f1': 0.0, 'invalid_rate': 1.0, 'seconds': 132.13748788833618, 'samples_per_second': 1.5135750133906856, 'label_id_style': 'default'}


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

qwen35_4b / tnews:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 6. Prepare supervised fine-tuning data

This converts CLUE classification rows into instruction-response text.

In [ ]:
def format_sft_example(task: str, ex: Dict[str, Any]) -> Dict[str, str]:
    prompt = build_prompt(task, ex)
    answer = label_id_to_name(task, ex["label"])
    return {"text": prompt + answer}


def make_sft_dataset(task: str, max_train_samples: int = MAX_TRAIN_SAMPLES):
    ds = load_dataset("clue", task, split="train")
    ds = ds.select(range(min(max_train_samples, len(ds))))
    return ds.map(lambda ex: format_sft_example(task, ex), remove_columns=ds.column_names)


sample_sft = make_sft_dataset("afqmc", 3)
print(sample_sft[0]["text"])


## 7. LoRA fine-tuning

Hugging Face PEFT LoRA training can be slower or more fragile on MPS than CUDA. Start with 200–1000 samples. If this fails on Mac, use a CUDA machine or switch to MLX-LM LoRA.

In [ ]:
def train_lora_for_task(model_key: str, model_id: str, task: str, out_dir: Optional[str] = None, max_train_samples: int = MAX_TRAIN_SAMPLES, epochs: float = 1.0):
    if not PEFT_AVAILABLE:
        raise RuntimeError("PEFT/TRL is not available. Re-run the install cell and restart kernel.")
    out_dir = out_dir or f"{ADAPTERS_DIR}/{model_key}_{task}_lora"
    os.makedirs(out_dir, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = dict(trust_remote_code=True)
    if DEVICE == "cuda":
        kwargs.update(torch_dtype=torch.float16, device_map="auto")
    elif DEVICE == "mps":
        kwargs.update(torch_dtype=torch.float16)
    else:
        kwargs.update(torch_dtype=torch.float32)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if DEVICE in ["mps", "cpu"]:
        model.to(DEVICE)
    train_ds = make_sft_dataset(task, max_train_samples)
    lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    training_args = TrainingArguments(output_dir=out_dir, per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=2e-4, num_train_epochs=epochs, logging_steps=10, save_steps=250, save_total_limit=2, fp16=(DEVICE == "cuda"), bf16=False, report_to="none", remove_unused_columns=False)
    trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_ds, peft_config=lora_config, tokenizer=tokenizer, dataset_text_field="text", max_seq_length=512)
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    del trainer, model
    cleanup_memory()
    return out_dir

In [ ]:
# Example: fine-tune both models on AFQMC.
# Uncomment when ready. Start with MAX_TRAIN_SAMPLES=200 for a smoke test.

# trained_adapters = {}
# for model_key, model_id in MODELS.items():
#     adapter_dir = train_lora_for_task(model_key=model_key, model_id=model_id, task="afqmc", max_train_samples=MAX_TRAIN_SAMPLES, epochs=1.0)
#     trained_adapters[(model_key, "afqmc")] = adapter_dir
#     print(model_key, adapter_dir)

## 8. Evaluate LoRA adapters

In [ ]:
def load_causal_lm_with_adapter(model_id: str, adapter_dir: str):
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = dict(trust_remote_code=True)
    if DEVICE == "cuda":
        kwargs.update(torch_dtype=torch.float16, device_map="auto")
    elif DEVICE == "mps":
        kwargs.update(torch_dtype=torch.float16)
    else:
        kwargs.update(torch_dtype=torch.float32)
    base = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if DEVICE in ["mps", "cpu"]:
        base.to(DEVICE)
    model = PeftModel.from_pretrained(base, adapter_dir)
    model.eval()
    return tokenizer, model


def evaluate_adapter_on_task(model_key: str, model_id: str, adapter_dir: str, task: str, split: str = SPLIT, max_samples: int = MAX_EVAL_SAMPLES):
    ds = load_task_dataset(task, split, max_samples)
    label_id_style = infer_label_id_style(task, ds)
    valid_ids = {label_name_to_id(task, name, label_id_style) for name in get_label_names(task)}

    tokenizer, model = load_causal_lm_with_adapter(model_id, adapter_dir)
    rows = []
    start = time.time()

    for ex in tqdm(ds, desc=f"{model_key}+LoRA / {task}"):
        prompt = build_prompt(task, ex)
        raw = generate_answer(tokenizer, model, prompt, max_new_tokens=4)
        pred_name = extract_label_name(task, raw)
        pred_id = label_name_to_id(task, pred_name, label_id_style)
        gold_id = get_gold_label(task, ex)
        gold_name = label_id_to_name(task, gold_id)

        rows.append({
            "model_key": model_key,
            "model_id": model_id,
            "adapter_dir": adapter_dir,
            "task": task,
            "split": split,
            "gold_id": gold_id,
            "gold_name": gold_name,
            "pred_id": pred_id,
            "pred_name": pred_name,
            "raw_output": raw,
            "prompt": prompt,
        })

    elapsed = time.time() - start
    del model
    cleanup_memory()

    df = pd.DataFrame(rows)
    valid_pred = df["pred_id"].where(df["pred_id"].isin(valid_ids), "__invalid__")
    summary = {
        "model_key": model_key + "+lora",
        "model_id": model_id,
        "adapter_dir": adapter_dir,
        "task": task,
        "split": split,
        "samples": len(df),
        "accuracy": accuracy_score(df["gold_id"], valid_pred),
        "macro_f1": f1_score(df["gold_id"], valid_pred, average="macro", zero_division=0),
        "invalid_rate": float((valid_pred == "__invalid__").mean()),
        "seconds": elapsed,
        "samples_per_second": len(df) / max(elapsed, 1e-9),
        "label_id_style": label_id_style,
    }
    return df, summary


In [ ]:
# Evaluate trained adapters. Uncomment after running fine-tuning.

# adapter_rows, adapter_summaries = [], []
# for (model_key, task), adapter_dir in trained_adapters.items():
#     df_task, summary = evaluate_adapter_on_task(model_key, MODELS[model_key], adapter_dir, task)
#     adapter_rows.append(df_task)
#     adapter_summaries.append(summary)
# pd.concat(adapter_rows, ignore_index=True).to_csv(f"{RESULTS_DIR}/adapter_predictions.csv", index=False)
# pd.DataFrame(adapter_summaries).to_csv(f"{RESULTS_DIR}/adapter_summary.csv", index=False)
# pd.DataFrame(adapter_summaries)

## 9. Compare results and inspect errors

In [ ]:
def load_if_exists(path):
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

baseline = load_if_exists(f"{RESULTS_DIR}/baseline_summary.csv")
adapter = load_if_exists(f"{RESULTS_DIR}/adapter_summary.csv")
comparison = pd.concat([baseline, adapter], ignore_index=True)
if not comparison.empty:
    display(comparison.sort_values(["task", "macro_f1"], ascending=[True, False]))
else:
    print("No result files yet. Run baseline and/or adapter evaluation cells first.")

In [ ]:
preds = load_if_exists(f"{RESULTS_DIR}/baseline_predictions.csv")
if not preds.empty:
    errors = preds[preds["gold_id"] != preds["pred_id"]].copy()
    display(errors[["model_key", "task", "gold_id", "gold_name", "pred_id", "pred_name", "raw_output", "prompt"]].head(20))
else:
    print("No baseline prediction file yet.")


## 10. Recommended experiment protocol

1. Run baseline zero-shot evaluation on all selected CLUE tasks.
2. Fine-tune one task at a time using the same number of training samples for both models.
3. Re-evaluate the matching task first, then optionally test transfer to other tasks.
4. Track `accuracy`, `macro_f1`, `invalid_rate`, `seconds`, and `samples_per_second`.
5. Keep raw predictions so label-format failures can be separated from understanding failures.